# Merging DVF, Rent Control and Planned Green Spaces

This notebook combines our three datasets:
dvf_paris_2025, rent_control_2024_2025.csv` and planned_green_spaces into one merged dataframe.

1. Load all three datasets
2. Filter rent control to 2025
3. Spatial join: assign each DVF property to a rent control zone and quartier
4. Add average reference rent per zone to each DVF property
5. Add planned green space info per arrondissement to each DVF property

## 1. Load the datasets

Upload the following files to this session before running all cells in the sesion:
* dvf_paris_2025_aggregated.csv
* api_rent_control_2025.csv
* planned_green_spaces.csv

To access files from Google Drive, you can mount your Drive to this Colab session using the following code:

Mounted at /content/drive


In [4]:
import pandas as pd
import json
import geopandas as gpd
from shapely.geometry import Point, shape

data_dir = '../data/'

df_dvf   = pd.read_csv(f'{data_dir}dvf_paris_2025_aggregated.csv')
df_rent  = pd.read_csv(f'{data_dir}api_rent_control_2025.csv')
df_green = pd.read_csv(f'{data_dir}planned_green_spaces.csv')

# Drop the unnamed index column that was added during export
if 'Unnamed: 0' in df_rent.columns:
    df_rent = df_rent.drop(columns='Unnamed: 0')

print('DVF rows:          ', len(df_dvf))
print('Rent control rows: ', len(df_rent))  # expected: 320 (80 quartiers x 4 room counts)
print('Planned green rows:', len(df_green))

DVF rows:           38551
Rent control rows:  320
Planned green rows: 71


## 2. Prepare rent control polygons for the spatial join

The rent control dataset has 4 rows per quartier: one per room count (1, 2, 3, 4 rooms).
For the spatial join we only need one polygon per quartier. The polygon shape
is the same for all room counts.

We keep one row per unique `quarter_id` and convert the `geo_shape` column
from a string back into a format that GeoPandas can work with.

In [5]:
# Keep one row per quartier. The polygon is the same across all room counts
df_quartiers = df_rent.drop_duplicates(subset='quarter_id').copy()

# geo_shape was saved as a JSON string.
# convert it back to a dictionary
df_quartiers['geo_shape'] = df_quartiers['geo_shape'].apply(json.loads)

# Convert the dictionary to a format shapely can work with
# The polygon coordinates are nested inside ['geometry']
df_quartiers['geometry'] = df_quartiers['geo_shape'].apply(
    lambda x: shape(x['geometry'])
)

# Build a GeoDataFrame with the polygon geometries
# crs='EPSG:4326' means standard lat/lon coordinates
gdf_zones = gpd.GeoDataFrame(
    df_quartiers[['zone_id', 'quarter_id', 'quarter_name', 'geometry']],
    geometry='geometry',
    crs='EPSG:4326'
)

print('Unique quartier polygons:', len(gdf_zones))  # expected: 80

Unique quartier polygons: 80


In [6]:
gdf_zones.head()

,zone_id,quarter_id,quarter_name,geometry
0,1,23,Notre-Dame-des-Champs,"POLYGON ((2.33676 48.84013, 2.33673 48.83965, ..."
4,1,25,Saint-Thomas-d'Aquin,"POLYGON ((2.32213 48.84925, 2.32054 48.84842, ..."
8,1,26,Invalides,"POLYGON ((2.31901 48.85174, 2.31903 48.8517, 2..."
12,1,27,Ecole-Militaire,"POLYGON ((2.32008 48.84818, 2.31936 48.84785, ..."
16,1,28,Gros-Caillou,"POLYGON ((2.30954 48.85396, 2.30646 48.85413, ..."


## 3. Spatial join: assign each DVF property to a quartier and zone

Each DVF property has a lon and lat coordinate.
Convert those into point-geometries and check which quartier polygon
each point falls inside (point-in-polygon) join.

After this join, each DVF row will have a `zone_id` and `quarter_name`.

In [7]:
# Filter DVF to clean rows only (data_quality_flag == 'ok')
df_dvf_ok = df_dvf[df_dvf['data_quality_flag'] == 'ok'].copy()

print('DVF rows (ok flag only):', len(df_dvf_ok))

# Convert each property's lon/lat into a Point geometry
# Point(lon, lat)
df_dvf_ok['geometry'] = df_dvf_ok.apply(
    lambda row: Point(row['lon'], row['lat']),
    axis=1
)

# Build a GeoDataFrame for the properties
gdf_properties = gpd.GeoDataFrame(df_dvf_ok, geometry='geometry', crs='EPSG:4326')

# Spatial join: find which quartier polygon each property point falls inside
# how= left
# keep all DVF properties, even if no polygon match is found
# predicate= within t
#he point must be inside the polygon
gdf_joined = gpd.sjoin(
    gdf_properties,
    gdf_zones,
    how       = 'left',
    predicate = 'within'
)

# Drop the geometry column which we no longer need
df_dvf_merged = pd.DataFrame(gdf_joined.drop(columns=['geometry', 'index_right']))

print('DVF rows after spatial join:     ', len(df_dvf_merged))
print('Properties without a zone match: ', df_dvf_merged['zone_id'].isna().sum())
print()
print(df_dvf_merged[['address', 'zone_id', 'quarter_name']].head(5))

DVF rows (ok flag only): 37720
DVF rows after spatial join:      37720
Properties without a zone match:  0

                        address  zone_id           quarter_name
0  41 RUE DES BOURDONNAIS 75001        5                 Halles
1      22 BD DE L HOPITAL 75005       10     Jardin-des-Plantes
2     1 RUE PAUL SEJOURNE 75006        1  Notre-Dame-des-Champs
4          51 AV DE SEGUR 75007        1        Ecole-Militaire
5          51 AV DE SEGUR 75007        1        Ecole-Militaire


## 4. Add reference rent per zone

The rent control dataset already has construction periods averaged and is filtered for
unfurnished values only  

Here we average across the 4 room count categories (1, 2, 3, and 4 or more rooms) to get
one representative reference rent per zone,
then merge it into the DVF dataframe.

In [8]:
# Average reference rent per zone across all room counts
df_avg_rent = (
    df_rent
    .groupby('zone_id')
    .agg({'reference_rent':'mean', 'min_rent':'min', 'max_rent':'max'})
    .round(2)
    .reset_index()
    .rename(columns={'reference_rent': 'avg_reference_rent'})
)

print(df_avg_rent)

# Merge into the DVF dataframe on zone_id
df_dvf_merged = df_dvf_merged.merge(
    df_avg_rent,
    on  = 'zone_id',
    how = 'left'    # keep all DVF rows
)

print('\nColumn added: avg_reference_rent')

    zone_id  avg_reference_rent  min_rent  max_rent
0         1               30.88      18.9      44.0
1         2               28.49      16.4      42.1
2         3               28.14      18.2      37.3
3         4               27.54      16.5      40.8
4         5               26.00      15.1      38.8
5         6               26.99      16.9      37.8
6         7               25.97      15.9      36.5
7         8               25.02      14.0      35.6
8         9               24.14      13.9      34.7
9        10               25.22      15.1      37.4
10       11               24.19      13.8      36.6
11       12               24.74      13.8      38.4
12       13               21.12      10.6      34.1
13       14               22.72      12.6      32.8

Column added: avg_reference_rent


In [9]:
df_dvf_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37720 entries, 0 to 37719
Data columns (total 34 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   year                37720 non-null  int64  
 1   transaction_date    37720 non-null  object 
 2   commune_code        37720 non-null  int64  
 3   section             37720 non-null  object 
 4   plot_number         37720 non-null  int64  
 5   transaction_number  37720 non-null  int64  
 6   transaction_key     37720 non-null  object 
 7   transaction_type    37720 non-null  object 
 8   property_value      37720 non-null  float64
 9   street_number       37720 non-null  float64
 10  street_type         37720 non-null  object 
 11  street_code         37720 non-null  object 
 12  street_name         37720 non-null  object 
 13  postal_code         37720 non-null  int64  
 14  commune             37720 non-null  object 
 15  department_code     37720 non-null  int64  
 16  lot_

In [10]:
# Verify that each reference is within min-max bounds
reference_rent_check = True
for index, row in df_dvf_merged.iterrows():
    if row['avg_reference_rent'] < row['min_rent'] or row['avg_reference_rent'] > row['max_rent']:
      print("One or more reference rent values outside of min-max bounds")
      reference_rent_check = False
      break
else:
    print("All reference rent values are within min-max bounds.")

All reference rent values are within min-max bounds.


## 5. Add planned green space info per arrondissement

The planned green spaces dataset has one row per project.
So it need first to be aggregated to arrondissement level, counting projects and
summing up the total added green area in m². Eventually merge into DVF.

DVF has a `commune_code` column that encodes the arrondissement:
Example: 7511767:  characters at position 2-4 = '117'  ->  117 - 100 = 17

In [11]:
# 1: derive arrondissement number from commune_code
df_dvf_merged['arrondissement'] = df_dvf_merged['commune_code'] - 100

# 2: aggregate planned green spaces by arrondissement
df_green_agg = (
    df_green
    .groupby('arrondissement')
    .agg(
        planned_projects     = ('project_name', 'count'),  # number of projects
        total_added_green_m2 = ('added_space_indicator', 'sum')    # total m² added
    )
    .reset_index()
)

print(df_green_agg)

# 3: merge into DVF on arrondissement
# Arrondissements with no planned projects will get NaN (fill with 0)
df_dvf_merged = df_dvf_merged.merge(
    df_green_agg,
    on  = 'arrondissement',
    how = 'left'
)

df_dvf_merged['planned_projects']     = df_dvf_merged['planned_projects'].fillna(0).astype(int)
df_dvf_merged['total_added_green_m2'] = df_dvf_merged['total_added_green_m2'].fillna(0)

print('\nColumns added: arrondissement, planned_projects, total_added_green_m2')

    arrondissement  planned_projects  total_added_green_m2
0                2                 1                1660.0
1                4                 2                3744.0
2                5                 2               16320.0
3                7                 2                2300.0
4                8                 7                7332.0
5               10                 2                 499.0
6               11                 5                1174.0
7               12                 8               59226.0
8               13                 7               17836.0
9               14                 6                4119.0
10              15                 3               17694.0
11              16                 3               20450.0
12              17                 4               14560.0
13              18                 7               28030.0
14              19                 5                5555.0
15              20                 7               43015

### Ensuring Arrondissement 21 values removed:

In [12]:
display(df_green[df_green['arrondissement'] == 21])

,Unnamed: 0,project_name,arrondissement,admin_sector,completion_date,operation_type,added_space_indicator,latitude,longitude


## 6. Final check and export

In [13]:
df_dvf_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37720 entries, 0 to 37719
Data columns (total 37 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   year                  37720 non-null  int64  
 1   transaction_date      37720 non-null  object 
 2   commune_code          37720 non-null  int64  
 3   section               37720 non-null  object 
 4   plot_number           37720 non-null  int64  
 5   transaction_number    37720 non-null  int64  
 6   transaction_key       37720 non-null  object 
 7   transaction_type      37720 non-null  object 
 8   property_value        37720 non-null  float64
 9   street_number         37720 non-null  float64
 10  street_type           37720 non-null  object 
 11  street_code           37720 non-null  object 
 12  street_name           37720 non-null  object 
 13  postal_code           37720 non-null  int64  
 14  commune               37720 non-null  object 
 15  department_code    

In [ ]:
df_dvf_merged.head()

In [ ]:
# Export the merged dataset
df_dvf_merged.to_csv('../data/dvf_paris_2025_merged.csv', index=False)
print('Saved: dvf_paris_2025_merged.csv')
print('Shape:', df_dvf_merged.shape)